<a href="https://colab.research.google.com/github/specM7/DSGP_Group_33_Brain_Tumor_Predictor/blob/Pituitary-Adenomas-Malindu-2425440/Pituitary%20Adenomas%20-%20Malindu_2425440.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# If running in Colab and torch/torchvision isn't installed yet, uncomment:
# !pip -q install torch torchvision

import os
import json
import numpy as np
import random
from PIL import Image, ImageEnhance
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

import joblib

# EfficientNet-B0 feature extractor (torchvision)
import torch
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

FEATURE_EXTRACTOR_NAME = "efficientnet_b0"
IMAGE_SIZE = 224


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
# Directories for training and testing data
# train_dir = '/content/drive/MyDrive/Training'
train_dir = '/content/drive/MyDrive/full_train'

# Define image size for EfficientNet-B0
IMAGE_SIZE = 224


# # Load and shuffle the train data
# train_paths = []
# train_labels = []
# train_paths = []
# train_labels = []

# for label in os.listdir(train_dir):
#     label_path = os.path.join(train_dir, label)

#     # # Skip if this entry is not a directory
#     # if os.path.isdir(label_path)== False:
#     #     continue
#     train_paths.append(label_path)
#     train_labels.append(label_path)
train_paths = []
train_labels = []

for fname in os.listdir(train_dir):
    fname_lower = fname.lower()

    if not fname_lower.endswith(('.jpg', '.jpeg', '.png')):
        continue

    if 'tr-pi_' in fname_lower:
        label = 'pi'
    # elif 'tr-no_' in fname_lower:
    #     label = 'no'
    else:
        label = 'no'  # skip unknown files

    train_paths.append(os.path.join(train_dir, fname))
    train_labels.append(label)
from sklearn.model_selection import train_test_split

X_paths_train, X_paths_test, y_train, y_test = train_test_split(
    train_paths,
    train_labels,
    test_size=0.25,
    stratify=train_labels,   # ensures mix of pi & no
    random_state=42
)

print(f"Loaded {len(train_paths)} images")
print("Class distribution:")
from collections import Counter
print(Counter(train_labels))


In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
import os

# Select random indices for 10 images
# random_indices = random.sample(range(len(train_paths)), 12)
num_images_to_show = min(12, len(train_paths))
random_indices = random.sample(range(len(train_paths)), num_images_to_show)

# Create a figure to display images in 2 rows
fig, axes = plt.subplots(3, 4, figsize=(15, 8))
axes = axes.ravel()

for i_display, idx_original in enumerate(random_indices):
    # Load image using the original index from train_paths
    img_path = train_paths[idx_original]
    img = Image.open(img_path)
    img = img.resize((224, 224))  # Resize to consistent size

    # Display image on the i_display-th axis
    axes[i_display].imshow(img)
    axes[i_display].axis('off')  # Hide axis
    # Display class label
    axes[i_display].set_title(f"Label: {train_labels[idx_original]}", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Light augmentation on PIL images (optional)
def augment_image_pil(img):
    img = img.convert('RGB')
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))
    return img

# Load images as PIL (torchvision weights/transforms will handle normalization)
def open_pil_images(paths, image_size=None, augment=False):
    images = []
    for path in paths:
        img = Image.open(path).convert('RGB')
        if image_size is not None:
            img = img.resize((image_size, image_size))
        if augment:
            img = augment_image_pil(img)
        images.append(img)
    return images

# Encoding labels (convert label names to integers)
def encode_label(labels):
    label_map = {'no': 0, 'pi': 1}
    return np.array([label_map[label] for label in labels])


In [ ]:
# EfficientNet-B0 feature extractor + optional fine-tuning
class PathLabelDataset(torch.utils.data.Dataset):
    def __init__(self, paths, labels, preprocess, augment=False):
        self.paths = paths
        self.labels = encode_label(labels)
        self.preprocess = preprocess
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.augment:
            img = augment_image_pil(img)
        img = self.preprocess(img)
        label = int(self.labels[idx])
        return img, label


# Load EfficientNet-B0 model for feature extraction
def create_efficientnet_b0_feature_extractor(device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    weights = EfficientNet_B0_Weights.DEFAULT
    preprocess = weights.transforms()

    model = efficientnet_b0(weights=weights)
    model.classifier = torch.nn.Identity()
    model.eval()
    model.to(device)

    return model, preprocess, device


# Fine-tune EfficientNet-B0 (unfreeze last N blocks) then convert to feature extractor
def fine_tune_efficientnet_b0(
    paths,
    labels,
    device=None,
    unfreeze_last=30,
    epochs=3,
    batch_size=16,
    lr=1e-4,
    weight_decay=1e-4
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    weights = EfficientNet_B0_Weights.DEFAULT
    preprocess = weights.transforms()

    model = efficientnet_b0(weights=weights)

    # Replace classifier for 2 classes
    if isinstance(model.classifier, torch.nn.Sequential):
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = torch.nn.Linear(in_features, 2)
    else:
        in_features = model.classifier.in_features
        model.classifier = torch.nn.Linear(in_features, 2)

    # Freeze all parameters
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze last N blocks of the backbone
    if unfreeze_last is None:
        unfreeze_last = 0
    n_unfreeze = min(max(int(unfreeze_last), 0), len(model.features))
    if n_unfreeze > 0:
        for layer in model.features[-n_unfreeze:]:
            for param in layer.parameters():
                param.requires_grad = True

    # Always train the classifier
    for param in model.classifier.parameters():
        param.requires_grad = True

    model.to(device)

    dataset = PathLabelDataset(paths, labels, preprocess, augment=True)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=(device == "cuda")
    )

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

        epoch_loss = running_loss / max(total, 1)
        epoch_acc = correct / max(total, 1)
        print(f"Fine-tune epoch {epoch + 1}/{epochs} | loss: {epoch_loss:.4f} | acc: {epoch_acc:.4f}")

    # Convert to feature extractor
    model.classifier = torch.nn.Identity()
    model.eval()

    return model, preprocess, device


# Feature extraction using EfficientNet-B0 (global pooled embedding)
def extract_efficientnet_features(paths, labels, feature_extractor, preprocess, device, batch_size=16, augment=False):
    all_features = []
    encoded_labels = encode_label(labels)

    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i + batch_size]
        batch_images = open_pil_images(batch_paths, augment=augment)

        batch_tensor = torch.stack([preprocess(img) for img in batch_images]).to(device)

        with torch.no_grad():
            feats = feature_extractor(batch_tensor).detach().cpu().numpy()

        all_features.append(feats)

    all_features = np.vstack(all_features)
    print(f"All features shape: {all_features.shape}")
    return all_features, encoded_labels


In [ ]:
# Create and train the hybrid model: EfficientNet-B0 feature extractor + (PCA) + Random Forest
def create_hybrid_model(
    use_pca=True,
    pca_variance=0.98,
    fine_tune=True,
    unfreeze_last=30,
    fine_tune_epochs=3,
    fine_tune_lr=1e-4,
    fine_tune_batch_size=16,
    rf_class_weight=None
):
    if rf_class_weight is None:
        rf_class_weight = {0: 1, 1: 3}

    if fine_tune:
        feature_extractor, preprocess, device = fine_tune_efficientnet_b0(
            X_paths_train,
            y_train,
            unfreeze_last=unfreeze_last,
            epochs=fine_tune_epochs,
            batch_size=fine_tune_batch_size,
            lr=fine_tune_lr
        )
    else:
        feature_extractor, preprocess, device = create_efficientnet_b0_feature_extractor()

    X_train_feat, y_train_enc = extract_efficientnet_features(
        X_paths_train, y_train, feature_extractor, preprocess, device, augment=True
    )

    X_test_feat, y_test_enc = extract_efficientnet_features(
        X_paths_test, y_test, feature_extractor, preprocess, device, augment=False
    )

    n_trees_options = [10, 50, 100]
    train_acc = []
    test_acc = []

    best_model = None
    best_test_acc = -1
    best_test_pred = None
    best_n_trees = None

    for n_trees in n_trees_options:
        rf = RandomForestClassifier(
            n_estimators=n_trees,
            max_depth=None,
            class_weight=rf_class_weight,
            random_state=42,
            n_jobs=-1
        )

        if use_pca:
            # If pca_variance is a float in (0, 1], sklearn requires svd_solver='full'
            if isinstance(pca_variance, float) and 0 < pca_variance <= 1:
                pca = PCA(n_components=pca_variance, svd_solver='full')
            else:
                pca = PCA(n_components=pca_variance, svd_solver='randomized', random_state=42)

            model = Pipeline([
                ('pca', pca),
                ('rf', rf)
            ])
        else:
            model = rf

        model.fit(X_train_feat, y_train_enc)

        train_pred = model.predict(X_train_feat)
        test_pred = model.predict(X_test_feat)

        train_acc.append(accuracy_score(y_train_enc, train_pred))
        test_acc.append(accuracy_score(y_test_enc, test_pred))

        if test_acc[-1] > best_test_acc:
            best_test_acc = test_acc[-1]
            best_model = model
            best_test_pred = test_pred
            best_n_trees = n_trees

        print(f"Trees: {n_trees}")
        print(f"  Train acc: {train_acc[-1]:.4f}")
        print(f"  Test acc : {test_acc[-1]:.4f}")

    plt.figure(figsize=(8, 5))
    plt.plot(n_trees_options, train_acc, marker='o', label='Training Accuracy')
    plt.plot(n_trees_options, test_acc, marker='o', label='Testing Accuracy')
    plt.xlabel("Number of Trees")
    plt.ylabel("Accuracy")
    plt.title("Learning Curve: EfficientNet-B0 + PCA + Random Forest" if use_pca else "Learning Curve: EfficientNet-B0 + Random Forest")
    plt.legend()
    plt.grid(True)
    plt.show()

    print(f"\nBest model: {best_n_trees} trees | Test acc: {best_test_acc:.4f}\n")

    label_map = {0: "no", 1: "pi"}
    for i in range(min(15, len(X_paths_test))):
        pred_label = label_map[best_test_pred[i]]
        true_label = y_test[i]
        print(f"Image: {os.path.basename(X_paths_test[i])} | Predicted: {pred_label} | Actual: {true_label}")

    # Evaluation metrics + confusion matrix on test set
    test_pred = best_model.predict(X_test_feat)
    test_proba = None
    if hasattr(best_model, "predict_proba"):
        test_proba = best_model.predict_proba(X_test_feat)[:, 1]

    precision = precision_score(y_test_enc, test_pred, zero_division=0)
    recall = recall_score(y_test_enc, test_pred, zero_division=0)
    accuracy = accuracy_score(y_test_enc, test_pred)
    roc_auc = roc_auc_score(y_test_enc, test_proba) if test_proba is not None else float("nan")

    print("\nTest Metrics")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall   : {recall:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")
    print(f"  Accuracy : {accuracy:.4f}")

    print("\nClassification Report")
    print(classification_report(y_test_enc, test_pred, target_names=["no", "pi"], zero_division=0))

    cm = confusion_matrix(y_test_enc, test_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["no", "pi"])
    disp.plot(cmap='Blues')
    plt.title("Confusion Matrix (Test Set)")
    plt.show()

    return best_model, feature_extractor


In [ ]:
# Predict using EfficientNet-B0 features + trained RandomForest/PCA pipeline
def predict_images(paths, feature_extractor, preprocess, device, rf_classifier, batch_size=16):
    all_predictions = []
    all_probabilities = []

    label_map = {0: "no", 1: "pi"}

    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i + batch_size]
        batch_images = open_pil_images(batch_paths, augment=False)

        batch_tensor = torch.stack([preprocess(img) for img in batch_images]).to(device)

        with torch.no_grad():
            feats = feature_extractor(batch_tensor).detach().cpu().numpy()

        batch_predictions = rf_classifier.predict(feats)
        batch_probabilities = rf_classifier.predict_proba(feats)

        decoded_predictions = [label_map[pred] for pred in batch_predictions]

        all_predictions.extend(decoded_predictions)
        all_probabilities.append(batch_probabilities)

    all_probabilities = np.vstack(all_probabilities)
    return all_predictions, all_probabilities

# Grad-CAM++ for EfficientNet-B0 using RF feature importances as weights
def get_efficientnet_target_layer(model):
    return model.features[-1]

def get_rf_feature_weights(rf_classifier, feature_dim, device):
    importances = None
    rf_model = rf_classifier
    pca = None

    if hasattr(rf_classifier, "named_steps"):
        rf_model = rf_classifier.named_steps.get("rf", rf_classifier)
        pca = rf_classifier.named_steps.get("pca")

    if hasattr(rf_model, "feature_importances_"):
        importances = rf_model.feature_importances_

    if pca is not None and hasattr(pca, "components_") and importances is not None:
        if importances.shape[0] == pca.components_.shape[0]:
            importances = importances @ pca.components_

    if importances is None or importances.shape[0] != feature_dim:
        weights = torch.ones(feature_dim, device=device)
    else:
        weights = torch.tensor(importances, dtype=torch.float32, device=device)
        weights = weights / (weights.abs().sum() + 1e-8)

    return weights

def make_gradcampp_heatmap(image_path, feature_extractor, preprocess, device, rf_classifier, target_layer=None):
    img = Image.open(image_path).convert('RGB')
    img_resized = img.resize((IMAGE_SIZE, IMAGE_SIZE))

    input_tensor = preprocess(img_resized).unsqueeze(0).to(device)
    input_tensor.requires_grad_(True)

    if target_layer is None:
        target_layer = get_efficientnet_target_layer(feature_extractor)

    activations = {}
    gradients = {}

    def forward_hook(_, __, output):
        activations["value"] = output

    def backward_hook(_, grad_input, grad_output):
        gradients["value"] = grad_output[0]

    handle_f = target_layer.register_forward_hook(forward_hook)
    handle_b = target_layer.register_full_backward_hook(backward_hook)

    feature_extractor.zero_grad()
    output = feature_extractor(input_tensor)
    weights = get_rf_feature_weights(rf_classifier, output.shape[-1], device)
    score = (output * weights).sum()
    score.backward()

    handle_f.remove()
    handle_b.remove()

    acts = activations["value"]
    grads = gradients["value"]

    grads_2 = grads ** 2
    grads_3 = grads ** 3
    sum_acts_grads3 = (acts * grads_3).sum(dim=(2, 3), keepdim=True)

    alpha = grads_2 / (2 * grads_2 + sum_acts_grads3 + 1e-8)
    weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)

    cam = (weights * acts).sum(dim=1, keepdim=True)
    cam = F.relu(cam)
    cam = cam[0, 0].detach().cpu().numpy()
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)

    return img_resized, cam

def overlay_heatmap(img, heatmap, alpha=0.45, colormap='jet'):
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_img = Image.fromarray(heatmap_uint8).resize(img.size)
    heatmap_color = plt.get_cmap(colormap)(np.array(heatmap_img) / 255.0)
    heatmap_color = np.uint8(heatmap_color[..., :3] * 255)
    heatmap_color_img = Image.fromarray(heatmap_color)
    return Image.blend(img.convert('RGB'), heatmap_color_img, alpha)


In [ ]:
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

def compute_training_accuracy(feature_extractor, preprocess, device, rf_classifier, paths, labels):
    features, encoded_labels = extract_efficientnet_features(paths, labels, feature_extractor, preprocess, device, augment=False)
    predictions = rf_classifier.predict(features)
    accuracy = accuracy_score(encoded_labels, predictions)
    return accuracy


In [ ]:
def plot_training_accuracy(accuracy):
    plt.figure(figsize=(6, 4))
    plt.plot([1], [accuracy], marker='o')
    plt.ylim(0, 1)
    plt.xticks([1], ['Training'])
    plt.ylabel('Accuracy')
    plt.title('Training Accuracy of Hybrid EfficientNet-B0 + Random Forest Model')
    plt.grid(True)
    plt.show()


In [ ]:
# Save only the sklearn classifier/pipeline and a small config

def save_model(rf_classifier, save_dir, feature_extractor_name=FEATURE_EXTRACTOR_NAME, feature_extractor_state_dict=None):
    os.makedirs(save_dir, exist_ok=True)

    joblib.dump(rf_classifier, os.path.join(save_dir, "rf_classifier.pkl"))

    # If a PCA step exists, save it explicitly for backend compatibility
    pca = None
    if hasattr(rf_classifier, "named_steps"):
        pca = rf_classifier.named_steps.get("pca")
    if pca is not None:
        joblib.dump(pca, os.path.join(save_dir, "pca.pkl"))

    if feature_extractor_state_dict is not None:
        torch.save(feature_extractor_state_dict, os.path.join(save_dir, "feature_extractor.pth"))

    cfg = {
        "feature_extractor": feature_extractor_name,
        "image_size": IMAGE_SIZE
    }

    with open(os.path.join(save_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f)

    print("Model saved successfully.")


In [ ]:
# Load the sklearn classifier/pipeline + config

def load_saved_model(save_dir):
    rf_classifier = joblib.load(os.path.join(save_dir, "rf_classifier.pkl"))

    cfg_path = os.path.join(save_dir, "config.json")
    feature_extractor_name = FEATURE_EXTRACTOR_NAME

    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        feature_extractor_name = cfg.get("feature_extractor", FEATURE_EXTRACTOR_NAME)

    print("Model loaded successfully.")
    return rf_classifier, feature_extractor_name


def load_feature_extractor(save_dir, device=None):
    feature_extractor, preprocess, device = create_efficientnet_b0_feature_extractor(device=device)
    ckpt_path = os.path.join(save_dir, "feature_extractor.pth")

    if os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, map_location=device)
        feature_extractor.load_state_dict(state)
        feature_extractor.eval()
        print("Loaded fine-tuned EfficientNet weights.")
    else:
        print("Using default EfficientNet weights.")

    return feature_extractor, preprocess, device


def load_pca(save_dir):
    pca_path = os.path.join(save_dir, "pca.pkl")
    if os.path.exists(pca_path):
        print("Loaded PCA transform.")
        return joblib.load(pca_path)
    print("No PCA transform found.")
    return None


In [ ]:
# STEP 1: Train the hybrid model (EfficientNet-B0 features + RF/PCA)
rf_classifier, feature_extractor = create_hybrid_model(
    use_pca=True,
    pca_variance=0.98,
    fine_tune=True,
    unfreeze_last=30,
    fine_tune_epochs=3,
    fine_tune_lr=1e-4,
    fine_tune_batch_size=16
)


In [ ]:
# STEP 2: Save model to Google Drive
MODEL_SAVE_PATH = "/content/drive/MyDrive/new_backend_hybrid_efficientnetb0_rf_model"

save_model(
    rf_classifier,
    MODEL_SAVE_PATH,
    FEATURE_EXTRACTOR_NAME,
    feature_extractor_state_dict=feature_extractor.state_dict()
)

# STEP 3: Load saved model
rf_classifier_loaded, feature_extractor_name_loaded = load_saved_model(MODEL_SAVE_PATH)


In [ ]:
# UPLOAD IMAGE AND PREDICT (EfficientNet-B0 features)

from google.colab import files
import os
from PIL import Image
import matplotlib.pyplot as plt

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print(f"Uploaded image: {image_path}")

img = Image.open(image_path)
plt.imshow(img)
plt.axis('off')
plt.title("Uploaded Image")
plt.show()

rf_classifier, feature_extractor_name = load_saved_model(MODEL_SAVE_PATH)
feature_extractor, preprocess, device = load_feature_extractor(MODEL_SAVE_PATH)

predictions, probabilities = predict_images(
    paths=[image_path],
    feature_extractor=feature_extractor,
    preprocess=preprocess,
    device=device,
    rf_classifier=rf_classifier
)

predicted_label = predictions[0]
tumor_probability = probabilities[0][1]

print("\n==================== PREDICTION ====================")
if predicted_label == 'pi':
    print(" PREDICTION: PITUITARY TUMOR DETECTED")
else:
    print(" PREDICTION: NO PITUITARY TUMOR DETECTED")
print(f"Confidence (Tumor Probability): {tumor_probability:.4f}")
print("====================================================")

# Grad-CAM++ heatmap
orig_img, heatmap = make_gradcampp_heatmap(image_path, feature_extractor, preprocess, device, rf_classifier)
overlay = overlay_heatmap(orig_img, heatmap, alpha=0.45)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(orig_img)
axes[0].axis('off')
axes[0].set_title("Original")

axes[1].imshow(overlay)
axes[1].axis('off')
axes[1].set_title("Grad-CAM++ Heatmap")

plt.tight_layout()
plt.show()


In [ ]:
# plt.figure(figsize=(8,4))
# plt.grid(True)
# plt.plot(history.history['sparse_categorical_accuracy'], '.g-', linewidth=2)
# plt.plot(history.history['loss'], '.r-', linewidth=2)
# plt.title('Model Training History')
# plt.xlabel('epoch')
# plt.xticks([x for x in range(epochs)])
# plt.legend(['Accuracy', 'Loss'], loc='upper left', bbox_to_anchor=(1, 1))
# plt.show()